# 3온도 물류센터 카메라 도메인 변화 증강 후보

냉장·냉동 구역 진입 시 카메라에 나타날 수 있는 저조도, 노이즈, 흐림, 렌즈 결로, 조명 반사 변화를 비교한다. 온도 자체를 물리적으로 재현하는 것이 아니라, 해당 환경에서 생기는 **영상 열화**에 강건한 객체 인식 데이터셋을 만들기 위한 후보 선택 단계다.

각 실행은 원본 1장과 후보 8장을 개별 PNG 및 3×3 비교판으로 저장한다. 메모리 사용을 낮추기 위해 아래 마지막 셀에서 한 그룹씩 실행한다.

In [ ]:
from pathlib import Path
import gc
import random

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_DIR = Path("../../dataset/raw_examples").resolve()
OUTPUT_DIR = Path("previews").resolve()
CANDIDATE_DIR = OUTPUT_DIR / "candidate_variants"
COMPARISON_DIR = OUTPUT_DIR / "library_comparisons"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print(f"원본 폴더: {RAW_DIR}")
print(f"개별 후보 저장 폴더: {CANDIDATE_DIR}")
print(f"비교판 저장 폴더: {COMPARISON_DIR}")
print(f"Torch: {torch.__version__} | MPS: {torch.backends.mps.is_available()}")
print(f"OpenCV: {cv2.__version__} | Albumentations: {A.__version__}")

## 그룹 의미

- `opencv`: 저조도·노이즈·흐림 같은 기본 카메라 열화
- `torchvision`: 전역 조명·대비·색 변화와 선명도/흐림 후보
- `albumentations`: 단일 변환 8종. 이후 segmentation mask와 함께 적용하기 좋은 후보
- `s2_condensation`: 렌즈 결로의 위치·범위·강도
- `s3_glare`: LED, 비닐 포장, 젖은 바닥 등의 국소 반사 하이라이트
- `s5_two_effects`: 강한 열화를 최대 두 개만 섞은 현실적 복합 후보

`RandomFog`는 물방울 결로가 아니라 균일한 haze 후보이다. 결로 자체는 S2의 커스텀 합성을 우선 비교한다.

In [ ]:
# 비교판용 축소 미리보기: 저장 원본의 해상도는 유지하고, 화면용 이미지만 줄인다.
def preview_rgb(image_bgr, max_width=480):
    height, width = image_bgr.shape[:2]
    if width > max_width:
        scale = max_width / width
        image_bgr = cv2.resize(image_bgr, (max_width, int(height * scale)))
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


# 저조도에서 ISO/게인이 높아질 때의 센서 읽기 노이즈를 근사한다.
# sigma가 클수록 입자가 굵고 강해진다.
def add_gaussian_noise(image_bgr, sigma=15):
    noise = np.random.normal(0, sigma, image_bgr.shape).astype(np.float32)
    return np.clip(image_bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)


# 자동 노출이 어두워진 저온 구역 내부를 근사한다. gamma < 1이면 더 어두워진다.
def adjust_gamma(image_bgr, gamma=0.7):
    table = np.array([((value / 255.0) ** (1.0 / gamma)) * 255 for value in range(256)]).astype(np.uint8)
    return cv2.LUT(image_bgr, table)


# 로봇/카메라 이동 중 셔터가 길어져 한 방향으로 번지는 현상을 근사한다.
# kernel_size는 번짐 길이, angle은 이동 방향이다.
def add_motion_blur(image_bgr, kernel_size=17, angle=0):
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    center = kernel_size // 2
    cv2.line(kernel, (0, center), (kernel_size - 1, center), 1, 1)
    rotation = cv2.getRotationMatrix2D((center, center), angle, 1.0)
    kernel = cv2.warpAffine(kernel, rotation, (kernel_size, kernel_size))
    kernel /= kernel.sum()
    return cv2.filter2D(image_bgr, -1, kernel)


# 천장 LED가 비닐 포장·금속·젖은 바닥·렌즈에 반사하는 국소 하이라이트를 근사한다.
# strength는 밝기, radius_ratio는 반사 영역 크기, center는 반사 위치다.
def add_glare(image_bgr, strength=0.55, radius_ratio=0.22, center=None):
    height, width = image_bgr.shape[:2]
    center = center or (int(width * 0.72), int(height * 0.25))
    x_grid, y_grid = np.meshgrid(np.arange(width), np.arange(height))
    distance = np.sqrt((x_grid - center[0]) ** 2 + (y_grid - center[1]) ** 2)
    sigma = radius_ratio * min(width, height)
    alpha = (strength * np.exp(-(distance ** 2) / (2 * sigma ** 2)))[..., None]
    return np.clip(image_bgr.astype(np.float32) * (1 - alpha) + 255 * alpha, 0, 255).astype(np.uint8)


# 상온에서 저온 구역으로 진입할 때 렌즈에 생기는 결로/성에 의한 국소 흐림을 근사한다.
# strength는 흐림 농도, radius_ratio는 결로 범위, center는 렌즈에서 뿌연 위치다.
def add_condensation(image_bgr, strength=0.7, radius_ratio=0.28, center=None):
    height, width = image_bgr.shape[:2]
    center = center or (int(width * 0.50), int(height * 0.45))
    x_grid, y_grid = np.meshgrid(np.arange(width), np.arange(height))
    distance = np.sqrt((x_grid - center[0]) ** 2 + (y_grid - center[1]) ** 2)
    sigma = radius_ratio * min(width, height)
    mask = (strength * np.exp(-(distance ** 2) / (2 * sigma ** 2)))[..., None]
    blurred = cv2.GaussianBlur(image_bgr, (0, 0), sigmaX=12)
    result = image_bgr.astype(np.float32) * (1 - mask) + blurred.astype(np.float32) * mask
    haze = 0.15 * mask
    return np.clip(result * (1 - haze) + 245 * haze, 0, 255).astype(np.uint8)


# 후보를 하나씩 생성·저장해 메모리 사용을 낮춘다. plt.show()는 호출하지 않아 팝업이 뜨지 않는다.
def save_candidate_group(target_stem, original_bgr, group_name, recipes):
    """원본과 후보 8개를 한 장씩 저장하고, 3×3 비교판도 파일로 저장한다."""
    assert len(recipes) == 8
    group_dir = CANDIDATE_DIR / target_stem / group_name
    group_dir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(group_dir / "00_original.png"), original_bgr)
    previews = [("original", preview_rgb(original_bgr))]

    for index, (name, recipe) in enumerate(recipes, start=1):
        image_bgr = recipe()
        cv2.imwrite(str(group_dir / f"{index:02d}_{name}.png"), image_bgr)
        previews.append((name, preview_rgb(image_bgr)))
        del image_bgr
        gc.collect()

    fig, axes = plt.subplots(3, 3, figsize=(12, 13))
    for axis, (name, preview) in zip(axes.ravel(), previews):
        axis.imshow(preview)
        axis.set_title(name, fontsize=10)
        axis.axis("off")
    fig.suptitle(f"{target_stem} | {group_name} | original + 8 candidates", fontsize=15)
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.92, wspace=0.02, hspace=0.08)
    grid_path = COMPARISON_DIR / f"{target_stem}__{group_name}_comparison.png"
    fig.savefig(grid_path, dpi=160, bbox_inches="tight")
    plt.close(fig)  # Notebook 화면 출력 없이 파일만 저장
    del previews
    gc.collect()
    return grid_path


In [ ]:
# 한 번에 하나만 실행: opencv, torchvision, albumentations, s2_condensation, s3_glare, s5_two_effects
GROUP_TO_RUN = "opencv"
TARGET_STEM = "pinky-pro_2"

original_bgr = cv2.imread(str(RAW_DIR / f"{TARGET_STEM}.png"))
assert original_bgr is not None, f"이미지를 읽을 수 없습니다: {TARGET_STEM}"
height, width = original_bgr.shape[:2]

def torch_recipe(transform, seed_offset):
    original_pil = Image.fromarray(cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB))
    def apply():
        torch.manual_seed(SEED + seed_offset)
        return cv2.cvtColor(np.array(transform(original_pil)), cv2.COLOR_RGB2BGR)
    return apply

def albu_recipe(transform, seed_offset):
    return lambda: A.Compose([transform], seed=SEED + seed_offset)(image=original_bgr)["image"]

recipes_by_group = {
    # OpenCV: 기본적인 저조도 카메라 열화의 강도 범위를 고른다.
    "opencv": [
        ("noise_mild_sigma8", lambda: add_gaussian_noise(original_bgr, 8)),  # 약한 저조도 센서 노이즈
        ("noise_strong_sigma25", lambda: add_gaussian_noise(original_bgr, 25)),  # 강한 ISO 노이즈 후보
        ("low_light_mild_gamma085", lambda: adjust_gamma(original_bgr, 0.85)),  # 조도가 조금 낮은 통로
        ("low_light_strong_gamma055", lambda: adjust_gamma(original_bgr, 0.55)),  # 조도가 매우 낮은 내부 구역
        ("motion_blur_short", lambda: add_motion_blur(original_bgr, 9, 18)),  # 짧은 이동 흔들림
        ("motion_blur_long", lambda: add_motion_blur(original_bgr, 25, 18)),  # 빠른 이동/긴 노출의 큰 번짐
        ("gaussian_blur", lambda: cv2.GaussianBlur(original_bgr, (0, 0), 1.4)),  # 초점 이탈 또는 약한 렌즈 흐림
        ("low_contrast", lambda: cv2.convertScaleAbs(original_bgr, alpha=0.72, beta=36)),  # 안개·낮은 동적 범위로 인한 밋밋한 대비
    ],
    # Torchvision: 카메라 자동 보정/후처리가 주는 전역적인 색감·선명도 변화를 고른다.
    "torchvision": [
        ("color_jitter_mild", torch_recipe(transforms.ColorJitter(0.15, 0.10, 0.05), 1)),  # 구역별 LED 색·노출의 작은 차이
        ("color_jitter_wide", torch_recipe(transforms.ColorJitter(0.35, 0.25, 0.15), 2)),  # 자동 화이트밸런스/노출이 크게 달라진 후보
        ("gaussian_blur_mild", torch_recipe(transforms.GaussianBlur(5, (0.3, 0.7)), 3)),  # 약한 초점 흐림
        ("gaussian_blur_strong", torch_recipe(transforms.GaussianBlur(13, (1.2, 2.2)), 4)),  # 더 강한 초점 이탈 후보
        ("jitter_plus_blur", torch_recipe(transforms.Compose([transforms.ColorJitter(0.30, 0.20, 0.10), transforms.GaussianBlur(9, (0.4, 1.2))]), 5)),  # 조명 변화와 흐림 동시 발생
        ("auto_contrast", torch_recipe(transforms.RandomAutocontrast(p=1.0), 6)),  # 카메라 자동 대비 보정 후보
        ("adjust_sharpness", torch_recipe(transforms.RandomAdjustSharpness(1.5, p=1.0), 7)),  # 과도한 선명도 후처리 후보
        ("posterize_5bits", torch_recipe(transforms.RandomPosterize(5, p=1.0), 8)),  # 낮은 색 단계/영상 처리 artefact 후보
    ],
    # Albumentations: 나중에 segmentation mask와 동기화하기 좋은 단일 변환 후보다.
    "albumentations": [
        ("brightness_contrast", albu_recipe(A.RandomBrightnessContrast(brightness_limit=(-0.35, 0.15), contrast_limit=(-0.15, 0.20), p=1.0), 1)),  # 저조도/노출·대비 변화
        ("gamma_low_light", albu_recipe(A.RandomGamma(gamma_limit=(120, 150), p=1.0), 2)),  # 감마 기반 어두운 노출
        ("gauss_noise", albu_recipe(A.GaussNoise(std_range=(0.01, 0.04), p=1.0), 3)),  # 일반 센서 노이즈
        ("iso_noise", albu_recipe(A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.15, 0.35), p=1.0), 4)),  # 고감도 카메라의 색 노이즈
        ("motion_blur", albu_recipe(A.MotionBlur(blur_limit=(7, 13), p=1.0), 5)),  # 이동 중 방향성 흐림
        ("gaussian_blur", albu_recipe(A.GaussianBlur(blur_limit=(5, 11), sigma_limit=(0.8, 2.0), p=1.0), 6)),  # 초점/렌즈의 비방향성 흐림
        ("image_compression", albu_recipe(A.ImageCompression(compression_type="jpeg", quality_range=(25, 50), p=1.0), 7)),  # 전송·저장 중 낮아진 영상 품질
        ("uniform_haze_fog", albu_recipe(A.RandomFog(fog_coef_range=(0.15, 0.30), alpha_coef=0.08, p=1.0), 8)),  # 저온 진입 직후 렌즈/안경 전체가 뿌연 haze 후보
    ],
    # S2: 국소 결로가 생기는 위치·면적·농도를 비교한다.
    "s2_condensation": [
        ("center_mild", lambda: add_condensation(original_bgr, 0.35, 0.18)),  # 중앙의 작고 약한 결로
        ("center_medium", lambda: add_condensation(original_bgr, 0.60, 0.28)),  # 중앙의 일반적인 결로
        ("center_strong", lambda: add_condensation(original_bgr, 0.80, 0.34)),  # 중앙 시야를 크게 가리는 심한 결로
        ("left_edge", lambda: add_condensation(original_bgr, 0.60, 0.26, (int(width * 0.18), int(height * 0.45)))),  # 렌즈 왼쪽 가장자리 결로
        ("right_edge", lambda: add_condensation(original_bgr, 0.60, 0.26, (int(width * 0.82), int(height * 0.45)))),  # 렌즈 오른쪽 가장자리 결로
        ("top_edge", lambda: add_condensation(original_bgr, 0.55, 0.24, (int(width * 0.50), int(height * 0.12)))),  # 상단에 맺힌 결로
        ("bottom_edge", lambda: add_condensation(original_bgr, 0.55, 0.24, (int(width * 0.50), int(height * 0.88)))),  # 하단에 맺힌 결로
        ("wide_haze", lambda: add_condensation(original_bgr, 0.42, 0.48)),  # 넓게 퍼진 옅은 결로막
    ],
    # S3: 반사의 크기·밝기·위치를 바꿔 실제 조명 반사에 가까운 후보를 고른다.
    "s3_glare": [
        ("small_mild", lambda: add_glare(original_bgr, 0.25, 0.10)),  # 작은 약한 반사점
        ("small_strong", lambda: add_glare(original_bgr, 0.75, 0.10)),  # 작은 강한 LED 반사점
        ("wide_soft", lambda: add_glare(original_bgr, 0.35, 0.34)),  # 넓고 부드러운 비닐/바닥 반사
        ("wide_strong", lambda: add_glare(original_bgr, 0.65, 0.30)),  # 넓고 밝은 과노출 반사
        ("upper_left", lambda: add_glare(original_bgr, 0.55, 0.18, (int(width * 0.20), int(height * 0.20)))),  # 좌상단 조명 반사
        ("upper_right", lambda: add_glare(original_bgr, 0.55, 0.18, (int(width * 0.80), int(height * 0.20)))),  # 우상단 조명 반사
        ("floor_left", lambda: add_glare(original_bgr, 0.45, 0.20, (int(width * 0.25), int(height * 0.78)))),  # 좌하단 젖은 바닥 반사
        ("floor_right", lambda: add_glare(original_bgr, 0.45, 0.20, (int(width * 0.75), int(height * 0.78)))),  # 우하단 젖은 바닥 반사
    ],
    # S5: 비현실적 과증강을 막기 위해 강한 열화를 최대 두 개만 조합한다.
    "s5_two_effects": [
        ("condensation_mild_plus_low_light", lambda: adjust_gamma(add_condensation(original_bgr, 0.40, 0.22), 0.80)),  # 약한 결로 + 약한 저조도
        ("condensation_medium_plus_low_light", lambda: adjust_gamma(add_condensation(original_bgr, 0.65, 0.30), 0.68)),  # 일반 결로 + 저조도
        ("edge_condensation_plus_low_light", lambda: adjust_gamma(add_condensation(original_bgr, 0.60, 0.25, (int(width * 0.18), int(height * 0.45))), 0.72)),  # 가장자리 결로 + 저조도
        ("condensation_plus_noise", lambda: add_gaussian_noise(add_condensation(original_bgr, 0.55, 0.26), 12)),  # 결로 + 저조도 노이즈
        ("condensation_plus_motion_blur", lambda: add_motion_blur(add_condensation(original_bgr, 0.52, 0.24), 13, 18)),  # 결로 + 이동 흐림
        ("glare_plus_low_light", lambda: adjust_gamma(add_glare(original_bgr, 0.48, 0.20), 0.72)),  # 반사 + 어두운 구역
        ("glare_plus_noise", lambda: add_gaussian_noise(add_glare(original_bgr, 0.48, 0.18), 12)),  # 반사 + 센서 노이즈
        ("low_light_plus_motion_blur", lambda: add_motion_blur(adjust_gamma(original_bgr, 0.65), 15, 18)),  # 저조도 + 이동 흐림
    ],
}

assert GROUP_TO_RUN in recipes_by_group, f"지원하지 않는 그룹: {GROUP_TO_RUN}"
grid_path = save_candidate_group(TARGET_STEM, original_bgr, GROUP_TO_RUN, recipes_by_group[GROUP_TO_RUN])
print(f"저장 완료: {grid_path}")
print(f"개별 후보 폴더: {grid_path.parent / GROUP_TO_RUN}")